# Inhale / exhale finder: overlay of two MRI frames (sagittal)

Stacks two NIfTI frames from a folder, with their file names on the image:
- **Colour**: A in magenta, B in green. Where both agree the image is grey; a liver dome that moved shows as a magenta/green fringe.
- **Blend**: B over A with an opacity slider (0 = only A, 1 = only B).

A head-feet motion curve over **all** frames suggests the frames where the anatomy is highest and lowest on screen.

**How to use**
1. Set the folder and the two file names in the next cell, then run all cells.
2. Put the head at the **top** of the image (*Flip vertical* if needed).
3. Pick a sagittal slice through the liver dome and click *Recompute curve for this slice*.
4. The liver is **lowest at end-inhale** and **highest at end-exhale**: check the suggested frames with the overlay and the guide line.

In [12]:
# ---- settings: change the file names here to compare other frames ----
FOLDER = "/volatile/Datasets/Varian_Motion/chum/9_11/mri/4d_reconstruction/small_Series0801/native"
FILE_A = "frame_0000.nii.gz"     # None = automatic suggestion (highest on screen)
FILE_B = "frame_0001.nii.gz"     # None = automatic suggestion (lowest on screen)
SAG_INDEX = None           # sagittal slice; None = middle slice
FLIP_VERTICAL = False      # True if the head appears at the bottom

In [13]:
import os
import re
from functools import lru_cache

import ipywidgets as widgets
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from IPython.display import clear_output, display

## Helpers
Frame sorting by number (`t_2` before `t_10`) and the head-feet motion estimate (weighted towards pixels that move over time, so the static spine does not dominate).

In [14]:
def natural_key(name: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]


def list_frames(folder: str) -> list[str]:
    files = sorted((f for f in os.listdir(folder) if f.endswith((".nii", ".nii.gz"))),
                   key=natural_key)
    if not files:
        raise FileNotFoundError(f"No .nii/.nii.gz files in {folder}")
    return files


def vertical_shift(img, ref, weight, max_shift):
    """Sub-pixel shift of img vs ref along rows (weighted NCC + parabola).
    Positive = content moved to HIGHER row indices."""
    H = img.shape[0]
    scores = []
    for s in range(-max_shift, max_shift + 1):
        if s >= 0:
            a, b, w = img[s:], ref[:H - s], weight[:H - s]
        else:
            a, b, w = img[:H + s], ref[-s:], weight[-s:]
        ws = w.sum() + 1e-8
        da, db = a - (a * w).sum() / ws, b - (b * w).sum() / ws
        num = (w * da * db).sum()
        den = np.sqrt((w * da * da).sum() * (w * db * db).sum()) + 1e-8
        scores.append(num / den)
    scores = np.array(scores)
    k = int(np.argmax(scores))
    off = 0.0
    if 0 < k < len(scores) - 1:
        y0, y1, y2 = scores[k - 1:k + 2]
        d = y0 - 2 * y1 + y2
        off = 0.5 * (y0 - y2) / d if abs(d) > 1e-12 else 0.0
    return (k - max_shift) + off


def motion_curve(slices: np.ndarray, max_shift: int = 15) -> np.ndarray:
    """slices [T, rows, cols] (rows = head-feet as displayed) -> shift per frame
    in rows vs the median frame, weighted towards pixels that change over time
    (liver / diaphragm) so static anatomy (spine) does not dominate."""
    ref = np.median(slices, axis=0)
    weight = slices.std(axis=0)
    weight = weight / (weight.max() + 1e-8)
    return np.array([vertical_shift(s, ref, weight, max_shift) for s in slices])

## Viewer

In [15]:
def show_overlay(folder: str, file_a: str | None = None, file_b: str | None = None,
                 sag_index: int | None = None, flip_vertical: bool = False,
                 window_percentiles=(1, 99), figsize=(7, 7)):
    names = list_frames(folder)
    n = len(names)

    @lru_cache(maxsize=128)
    def load(name: str):
        img = nib.as_closest_canonical(nib.load(os.path.join(folder, name)))   # (R, A, S)
        data = np.asarray(img.dataobj, dtype=np.float32)
        if data.ndim == 4:
            data = data[..., 0]
        return data, tuple(float(z) for z in img.header.get_zooms()[:3])

    data0, zooms = load(names[0])
    n_sag = data0.shape[0]
    sag0 = n_sag // 2 if sag_index is None else int(sag_index)

    # ---- motion curve over all frames (for the chosen sagittal slice) ---------------
    def sagittal(name: str, k: int) -> np.ndarray:
        return load(name)[0][min(k, n_sag - 1)].T          # rows = S (bottom->top), cols = A

    def compute_curve(k: int):
        print(f"computing head-feet motion over {n} frames (sagittal {k}) ...")
        slices = np.stack([sagittal(nm, k) for nm in names])
        return motion_curve(slices) * zooms[2]               # mm; + = higher row = higher on screen

    curve = {"k": sag0, "mm": compute_curve(sag0)}
    sample = np.concatenate([load(names[i])[0].ravel()
                             for i in np.unique(np.linspace(0, n - 1, min(n, 20)).astype(int))])
    vmin, vmax = np.percentile(sample, window_percentiles)

    def extremes(flipped: bool):
        mm = -curve["mm"] if flipped else curve["mm"]
        return int(np.argmax(mm)), int(np.argmin(mm))        # (highest on screen, lowest on screen)

    hi0, lo0 = extremes(flip_vertical)
    file_a = file_a if file_a in names else names[hi0]
    file_b = file_b if file_b in names else names[lo0]
    print(f"suggested: highest on screen (exhale if head is up) = {names[hi0]}, "
          f"lowest (inhale) = {names[lo0]}")

    # ---- widgets ----------------------------------------------------------------------
    dd_a = widgets.Dropdown(options=names, value=file_a, description="A (magenta)")
    dd_b = widgets.Dropdown(options=names, value=file_b, description="B (green)")
    mode = widgets.ToggleButtons(options=["Colour", "Blend"], value="Colour")
    opacity = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="B opacity")
    sag = widgets.IntSlider(value=sag0, min=0, max=n_sag - 1, description="Sagittal",
                            continuous_update=False)
    flip_v = widgets.Checkbox(value=flip_vertical, description="Flip vertical")
    flip_h = widgets.Checkbox(value=False, description="Flip horizontal")
    rows = data0.shape[2]
    guide = widgets.IntSlider(value=rows // 2, min=0, max=rows - 1, description="Guide line")
    btn_hi = widgets.Button(description="A = highest")
    btn_lo = widgets.Button(description="B = lowest")
    btn_curve = widgets.Button(description="Recompute curve for this slice")
    out = widgets.Output()

    def prep(img):
        img = np.clip((img - vmin) / (vmax - vmin + 1e-8), 0, 1)
        if flip_v.value:
            img = img[::-1]
        if flip_h.value:
            img = img[:, ::-1]
        return img

    def render(*_):
        a, b = prep(sagittal(dd_a.value, sag.value)), prep(sagittal(dd_b.value, sag.value))
        ia, ib = names.index(dd_a.value), names.index(dd_b.value)
        with out:
            clear_output(wait=True)
            fig, (ax, axc) = plt.subplots(2, 1, figsize=figsize,
                                          gridspec_kw={"height_ratios": [4, 1]})
            if mode.value == "Colour":
                rgb = np.stack([a, b, a], axis=-1)
                ax.imshow(rgb, origin="lower", interpolation="nearest",
                          aspect=zooms[2] / zooms[1])
                label = f"A (magenta): {dd_a.value}\nB (green): {dd_b.value}"
            else:
                mix = (1 - opacity.value) * a + opacity.value * b
                ax.imshow(mix, cmap="gray", vmin=0, vmax=1, origin="lower",
                          interpolation="nearest", aspect=zooms[2] / zooms[1])
                label = (f"A: {dd_a.value} ({1 - opacity.value:.0%})\n"
                         f"B: {dd_b.value} ({opacity.value:.0%})")
            ax.axhline(guide.value, color="yellow", lw=0.8, ls="--")
            ax.text(0.01, 0.99, label, transform=ax.transAxes, va="top", ha="left",
                    color="white", fontsize=9,
                    bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
            ax.set_title(f"sagittal {sag.value}/{n_sag - 1}", fontsize=10)
            ax.axis("off")

            mm = -curve["mm"] if flip_v.value else curve["mm"]
            axc.plot(mm, lw=0.8, color="gray")
            axc.plot(ia, mm[ia], "o", color="magenta", label="A")
            axc.plot(ib, mm[ib], "o", color="green", label="B")
            axc.set_xlabel("frame")
            axc.set_ylabel("height (mm)")
            axc.set_title(f"head-feet motion, sagittal {curve['k']} (up = higher on screen)",
                          fontsize=9)
            axc.legend(loc="upper right", fontsize=8)
            fig.tight_layout()
            plt.show()

    def set_extreme(which):
        hi, lo = extremes(flip_v.value)
        if which == "hi":
            dd_a.value = names[hi]
        else:
            dd_b.value = names[lo]

    def recompute(_):
        curve["k"], curve["mm"] = sag.value, compute_curve(sag.value)
        render()

    btn_hi.on_click(lambda _: set_extreme("hi"))
    btn_lo.on_click(lambda _: set_extreme("lo"))
    btn_curve.on_click(recompute)
    for w in (dd_a, dd_b, mode, opacity, sag, flip_v, flip_h, guide):
        w.observe(render, names="value")

    display(widgets.VBox([
        widgets.HBox([dd_a, dd_b]),
        widgets.HBox([mode, opacity]),
        widgets.HBox([sag, guide]),
        widgets.HBox([flip_v, flip_h]),
        widgets.HBox([btn_hi, btn_lo, btn_curve]),
    ]), out)
    render()

## Run

In [ ]:
show_overlay(FOLDER, file_a=FILE_A, file_b=FILE_B, sag_index=SAG_INDEX, flip_vertical=FLIP_VERTICAL)

computing head-feet motion over 594 frames (sagittal 16) ...
suggested: highest on screen (exhale if head is up) = frame_0189.nii.gz, lowest (inhale) = frame_0012.nii.gz


Output()